# Reinforcement Learning for Global Equity ETF Allocation

This notebook implements a PPO-based RL agent that dynamically allocates across global equity ETFs using macro signals.

**Key Features:**
- State: Momentum features from TLT, GLD, USO, SLV, VIXY, IBIT
- Actions: Allocate to SPY, EWJ, INDA, MCHI, EZU, or BIL (cash)
- Reward: Risk-adjusted returns (Sharpe-like)
- Algorithm: Proximal Policy Optimization (PPO)

In [ ]:
import pandas as pd
import datetime as dt
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
from typing import Callable, Dict
from trading_engine.core import (
    read_data, create_model_state, orchestrate_model_backtests,
    orchestrate_model_simulations, orchestrate_portfolio_backtests,
    orchestrate_portfolio_simulations
)
from trading_engine.models import MODELS
from trading_engine.optimizers import OPTIMIZERS
from common.constants import ProcessingMode

# RL imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

## 1. PPO Policy Network

In [ ]:
class PolicyNetwork(nn.Module):
    """Neural network that maps state features to action probabilities."""
    
    def __init__(self, state_dim: int, action_dim: int, hidden_dim: int = 128):
        super(PolicyNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        self.relu = nn.ReLU()
        self.softmax = nn.Softmax(dim=-1)
        
    def forward(self, state):
        x = self.relu(self.fc1(state))
        x = self.relu(self.fc2(x))
        action_probs = self.softmax(self.fc3(x))
        return action_probs


class ValueNetwork(nn.Module):
    """Neural network that estimates state values for advantage calculation."""
    
    def __init__(self, state_dim: int, hidden_dim: int = 128):
        super(ValueNetwork, self).__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, 1)
        self.relu = nn.ReLU()
        
    def forward(self, state):
        x = self.relu(self.fc1(state))
        x = self.relu(self.fc2(x))
        value = self.fc3(x)
        return value

## 2. PPO Agent

In [ ]:
class PPOAgent:
    """Proximal Policy Optimization agent for ETF allocation."""
    
    def __init__(self, state_dim: int, action_dim: int, 
                 lr_policy: float = 3e-4, lr_value: float = 1e-3,
                 gamma: float = 0.95, epsilon_clip: float = 0.2,
                 hidden_dim: int = 128):
        self.gamma = gamma
        self.epsilon_clip = epsilon_clip
        self.action_dim = action_dim
        
        # Networks
        self.policy = PolicyNetwork(state_dim, action_dim, hidden_dim)
        self.value = ValueNetwork(state_dim, hidden_dim)
        
        # Optimizers
        self.policy_optimizer = optim.Adam(self.policy.parameters(), lr=lr_policy)
        self.value_optimizer = optim.Adam(self.value.parameters(), lr=lr_value)
        
        # Storage for trajectories
        self.states = []
        self.actions = []
        self.rewards = []
        self.action_probs = []
        self.is_terminals = []
        
    def select_action(self, state, training=True):
        """Select action based on current policy."""
        state_tensor = torch.FloatTensor(state).unsqueeze(0)
        action_probs = self.policy(state_tensor)
        
        if training:
            # Sample from distribution during training (exploration)
            dist = Categorical(action_probs)
            action = dist.sample()
        else:
            # Choose best action during inference (exploitation)
            action = torch.argmax(action_probs, dim=1)
        
        return action.item(), action_probs[0, action.item()].item()
    
    def store_transition(self, state, action, reward, action_prob, is_terminal):
        """Store experience tuple."""
        self.states.append(state)
        self.actions.append(action)
        self.rewards.append(reward)
        self.action_probs.append(action_prob)
        self.is_terminals.append(is_terminal)
    
    def compute_advantages(self):
        """Compute GAE (Generalized Advantage Estimation)."""
        states_tensor = torch.FloatTensor(np.array(self.states))
        values = self.value(states_tensor).detach().numpy().flatten()
        
        advantages = []
        returns = []
        
        # Compute discounted returns and advantages
        gae = 0
        next_value = 0
        
        for t in reversed(range(len(self.rewards))):
            if self.is_terminals[t]:
                next_value = 0
                gae = 0
            
            # TD error: δ = r + γV(s') - V(s)
            delta = self.rewards[t] + self.gamma * next_value - values[t]
            # GAE: A = δ + γλδ' + ...
            gae = delta + self.gamma * 0.95 * gae  # λ=0.95
            
            advantages.insert(0, gae)
            returns.insert(0, gae + values[t])
            next_value = values[t]
        
        # Normalize advantages
        advantages = np.array(advantages)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        return advantages, np.array(returns)
    
    def update(self, epochs=10, batch_size=64):
        """Update policy and value networks using PPO."""
        if len(self.states) == 0:
            return
        
        # Compute advantages
        advantages, returns = self.compute_advantages()
        
        # Convert to tensors
        states_tensor = torch.FloatTensor(np.array(self.states))
        actions_tensor = torch.LongTensor(np.array(self.actions))
        old_probs_tensor = torch.FloatTensor(np.array(self.action_probs))
        advantages_tensor = torch.FloatTensor(advantages)
        returns_tensor = torch.FloatTensor(returns)
        
        # PPO update for multiple epochs
        for _ in range(epochs):
            # Mini-batch update
            indices = np.arange(len(self.states))
            np.random.shuffle(indices)
            
            for start in range(0, len(self.states), batch_size):
                end = min(start + batch_size, len(self.states))
                batch_indices = indices[start:end]
                
                # Get batch
                batch_states = states_tensor[batch_indices]
                batch_actions = actions_tensor[batch_indices]
                batch_old_probs = old_probs_tensor[batch_indices]
                batch_advantages = advantages_tensor[batch_indices]
                batch_returns = returns_tensor[batch_indices]
                
                # Evaluate current policy
                action_probs = self.policy(batch_states)
                dist = Categorical(action_probs)
                new_probs = dist.log_prob(batch_actions).exp()
                
                # Ratio for PPO
                ratio = new_probs / (batch_old_probs + 1e-8)
                
                # PPO clipped objective
                surr1 = ratio * batch_advantages
                surr2 = torch.clamp(ratio, 1 - self.epsilon_clip, 1 + self.epsilon_clip) * batch_advantages
                policy_loss = -torch.min(surr1, surr2).mean()
                
                # Value loss
                value_pred = self.value(batch_states).squeeze()
                value_loss = nn.MSELoss()(value_pred, batch_returns)
                
                # Update policy
                self.policy_optimizer.zero_grad()
                policy_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
                self.policy_optimizer.step()
                
                # Update value
                self.value_optimizer.zero_grad()
                value_loss.backward()
                torch.nn.utils.clip_grad_norm_(self.value.parameters(), 0.5)
                self.value_optimizer.step()
        
        # Clear trajectories
        self.clear_memory()
    
    def clear_memory(self):
        """Clear stored trajectories."""
        self.states = []
        self.actions = []
        self.rewards = []
        self.action_probs = []
        self.is_terminals = []

## 3. RL Model Wrapper for Trading Engine

In [ ]:
def RLGlobalEquityModel(
    trade_tickers: list,  # ["SPY-US", "EWJ-US", "INDA-US", "MCHI-US", "EZU-US", "BIL-US"]
    signal_tickers: list,  # ["TLT-US", "GLD-US", "USO-US", "SLV-US", "VIXY-US", "IBIT-US"]
    momentum_columns: list,  # e.g., ["close_momentum_10", "close_momentum_20", ...]
    trained_agent: PPOAgent = None,
    volatility_penalty: float = 0.5,  # λ in reward function
) -> Callable[[pl.LazyFrame], pl.LazyFrame]:
    """
    RL-based model that allocates across global equity ETFs based on macro signals.
    
    If trained_agent is None, returns equal-weight allocation (for training baseline).
    If trained_agent is provided, uses the trained policy to generate allocations.
    """
    
    def run_model(lf: pl.LazyFrame) -> pl.LazyFrame:
        # Collect the lazy frame
        df = lf.collect()
        
        # Get unique dates
        dates = df.select('date').unique().sort('date')
        
        # Initialize weights storage
        weights_list = []
        
        for date_row in dates.iter_rows():
            date = date_row[0]
            
            # Extract state: momentum features from signal tickers
            state_features = []
            date_df = df.filter(pl.col('date') == date)
            
            for signal_ticker in signal_tickers:
                ticker_df = date_df.filter(pl.col('ticker') == signal_ticker)
                
                for momentum_col in momentum_columns:
                    if len(ticker_df) > 0 and momentum_col in ticker_df.columns:
                        val = ticker_df.select(momentum_col).to_numpy()[0, 0]
                        state_features.append(val if not np.isnan(val) else 0.0)
                    else:
                        state_features.append(0.0)
            
            # Convert to numpy array
            state = np.array(state_features, dtype=np.float32)
            
            # Get action from agent or use equal weight
            if trained_agent is not None:
                action_idx, _ = trained_agent.select_action(state, training=False)
                # Create one-hot allocation
                weights = {ticker: 0.0 for ticker in trade_tickers}
                weights[trade_tickers[action_idx]] = 1.0
            else:
                # Equal weight baseline
                weights = {ticker: 1.0 / len(trade_tickers) for ticker in trade_tickers}
            
            # Store weights
            weight_dict = {'date': date}
            weight_dict.update(weights)
            weights_list.append(weight_dict)
        
        # Convert to polars DataFrame
        weights_df = pl.DataFrame(weights_list)
        
        return weights_df.lazy()
    
    return run_model

## 4. Training Loop

In [ ]:
def train_rl_agent(
    model_state: pl.LazyFrame,
    prices: pl.LazyFrame,
    trade_tickers: list,
    signal_tickers: list,
    momentum_columns: list,
    num_episodes: int = 100,
    update_frequency: int = 2000,  # Update every N timesteps
    volatility_penalty: float = 0.5,
    volatility_window: int = 20,
) -> PPOAgent:
    """
    Train PPO agent on historical data.
    """
    # Initialize agent
    state_dim = len(signal_tickers) * len(momentum_columns)
    action_dim = len(trade_tickers)
    agent = PPOAgent(state_dim, action_dim)
    
    # Collect data
    df = model_state.collect()
    prices_df = prices.collect()
    dates = df.select('date').unique().sort('date')['date'].to_list()
    
    print(f"Training on {len(dates)} days of data...")
    print(f"State dimension: {state_dim}, Action dimension: {action_dim}")
    
    # Training loop
    episode_rewards = []
    
    for episode in range(num_episodes):
        total_reward = 0
        timestep = 0
        
        # Track portfolio returns for volatility calculation
        recent_returns = []
        
        for i in range(len(dates) - 1):
            date = dates[i]
            next_date = dates[i + 1]
            
            # Extract state
            state_features = []
            date_df = df.filter(pl.col('date') == date)
            
            for signal_ticker in signal_tickers:
                ticker_df = date_df.filter(pl.col('ticker') == signal_ticker)
                for momentum_col in momentum_columns:
                    if len(ticker_df) > 0 and momentum_col in ticker_df.columns:
                        val = ticker_df.select(momentum_col).to_numpy()[0, 0]
                        state_features.append(val if not np.isnan(val) else 0.0)
                    else:
                        state_features.append(0.0)
            
            state = np.array(state_features, dtype=np.float32)
            
            # Select action
            action_idx, action_prob = agent.select_action(state, training=True)
            selected_ticker = trade_tickers[action_idx]
            
            # Get return for selected asset
            current_price_df = prices_df.filter(
                (pl.col('date') == date) & (pl.col('ticker') == selected_ticker)
            )
            next_price_df = prices_df.filter(
                (pl.col('date') == next_date) & (pl.col('ticker') == selected_ticker)
            )
            
            if len(current_price_df) > 0 and len(next_price_df) > 0:
                current_price = current_price_df.select('close').to_numpy()[0, 0]
                next_price = next_price_df.select('close').to_numpy()[0, 0]
                daily_return = (next_price - current_price) / current_price
            else:
                daily_return = 0.0
            
            recent_returns.append(daily_return)
            if len(recent_returns) > volatility_window:
                recent_returns.pop(0)
            
            # Compute reward: return - λ * volatility
            if len(recent_returns) >= 2:
                volatility = np.std(recent_returns)
                reward = daily_return - volatility_penalty * volatility
            else:
                reward = daily_return
            
            # Store transition
            is_terminal = (i == len(dates) - 2)
            agent.store_transition(state, action_idx, reward, action_prob, is_terminal)
            
            total_reward += daily_return  # Track actual returns for logging
            timestep += 1
            
            # Update agent periodically
            if timestep % update_frequency == 0:
                agent.update()
        
        # Final update at end of episode
        agent.update()
        
        episode_rewards.append(total_reward)
        
        if (episode + 1) % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            print(f"Episode {episode + 1}/{num_episodes}, Avg Reward (last 10): {avg_reward:.4f}")
    
    # Plot training progress
    plt.figure(figsize=(10, 5))
    plt.plot(episode_rewards)
    plt.title('Training Progress: Episode Returns')
    plt.xlabel('Episode')
    plt.ylabel('Total Return')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    return agent

## 5. Configuration and Data Loading

In [ ]:
# Experiment config
universe = [
    "SPY-US", "SLV-US", "GLD-US", "TLT-US", "USO-US", "UNG-US", "IXJ-US",
    "KXI-US", "JXI-US", "IXG-US", "IXN-US", "RXI-US", "MXI-US", "EXI-US",
    "IXC-US", "IEI-US", "SHY-US", "BIL-US", "JPXN-US", "INDA-US", "MCHI-US",
    "EZU-US", "IBIT-US", "ETHA-US", "VIXY-US", "EWJ-US"
]

# Trade tickers (actions)
trade_tickers = ["SPY-US", "EWJ-US", "INDA-US", "MCHI-US", "EZU-US", "BIL-US"]

# Signal tickers (state)
signal_tickers = ["TLT-US", "GLD-US", "USO-US", "SLV-US", "VIXY-US", "IBIT-US"]

# Features
momentum_columns = ["close_momentum_10", "close_momentum_20", "close_momentum_60"]

initial_value = 1_000_000
start_date = dt.date(2004, 1, 2)
end_date = dt.date(2025, 9, 10)

In [ ]:
# Load data and create model state
lf = read_data()
model_state, prices = create_model_state(
    lf=lf,
    features=momentum_columns,
    start_date=start_date,
    end_date=end_date,
    universe=universe
)

print("Data loaded successfully!")

## 6. Train the RL Agent

In [ ]:
# Train the agent
trained_agent = train_rl_agent(
    model_state=model_state,
    prices=prices,
    trade_tickers=trade_tickers,
    signal_tickers=signal_tickers,
    momentum_columns=momentum_columns,
    num_episodes=50,  # Adjust based on your needs
    update_frequency=2000,
    volatility_penalty=0.5,
)

print("\nTraining completed!")

## 7. Create Model Registry and Run Backtests

In [ ]:
# Create model registry
models_registry = {
    "RL_GlobalEquity": {
        "tickers": trade_tickers + signal_tickers,
        "columns": momentum_columns,
        "function": RLGlobalEquityModel(
            trade_tickers=trade_tickers,
            signal_tickers=signal_tickers,
            momentum_columns=momentum_columns,
            trained_agent=trained_agent,
            volatility_penalty=0.5,
        ),
    },
    "Baseline_EqualWeight": {
        "tickers": trade_tickers + signal_tickers,
        "columns": momentum_columns,
        "function": RLGlobalEquityModel(
            trade_tickers=trade_tickers,
            signal_tickers=signal_tickers,
            momentum_columns=momentum_columns,
            trained_agent=None,  # Equal weight baseline
            volatility_penalty=0.5,
        ),
    },
}

models = ["RL_GlobalEquity", "Baseline_EqualWeight"]

In [ ]:
# Run model backtests
model_insights = orchestrate_model_backtests(
    model_state=model_state,
    models=models,
    universe=universe,
    registry=models_registry
)

model_simulations = orchestrate_model_simulations(
    prices=prices,
    model_insights=model_insights
)

print("\nBacktest Results:")
for model_name in models:
    print(f"\n{model_name}:")
    print(model_simulations[model_name]['backtest_metrics'])

## 8. Visualize Results

In [ ]:
# Build equity curves
def build_equity_curve_multi(prices_df, model_simulations, model_names, trade_tickers):
    """Build equity curves for multiple models."""
    prices_collected = prices_df.collect() if hasattr(prices_df, 'collect') else prices_df
    
    curves = {}
    
    for model_name in model_names:
        weights_df = model_simulations[model_name]['backtest_weights']
        if hasattr(weights_df, 'collect'):
            weights_df = weights_df.collect()
        
        # Get portfolio returns
        dates = weights_df.select('date').unique().sort('date')['date'].to_list()
        portfolio_returns = []
        
        for i in range(len(dates) - 1):
            date = dates[i]
            next_date = dates[i + 1]
            
            # Get weights for this date
            weight_row = weights_df.filter(pl.col('date') == date)
            
            # Calculate portfolio return
            portfolio_return = 0.0
            for ticker in trade_tickers:
                if ticker in weight_row.columns:
                    weight = weight_row.select(ticker).to_numpy()[0, 0]
                    
                    # Get price return
                    current_price = prices_collected.filter(
                        (pl.col('date') == date) & (pl.col('ticker') == ticker)
                    ).select('close')
                    next_price = prices_collected.filter(
                        (pl.col('date') == next_date) & (pl.col('ticker') == ticker)
                    ).select('close')
                    
                    if len(current_price) > 0 and len(next_price) > 0:
                        ret = (next_price.to_numpy()[0, 0] - current_price.to_numpy()[0, 0]) / current_price.to_numpy()[0, 0]
                        portfolio_return += weight * ret
            
            portfolio_returns.append(portfolio_return)
        
        # Build equity curve
        equity = [1.0]
        for ret in portfolio_returns:
            equity.append(equity[-1] * (1 + ret))
        
        curves[model_name] = {
            'dates': dates[:len(equity)],
            'equity': equity
        }
    
    return curves


# Build and plot equity curves
equity_curves = build_equity_curve_multi(prices, model_simulations, models, trade_tickers)

plt.figure(figsize=(14, 7))
for model_name in models:
    plt.plot(equity_curves[model_name]['dates'], 
             equity_curves[model_name]['equity'], 
             label=model_name, linewidth=2)

plt.title("RL Global Equity Model vs Baseline", fontsize=14, fontweight='bold')
plt.xlabel("Date")
plt.ylabel("Cumulative Growth of $1")
plt.yscale('log')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Analyze Allocation Patterns

In [ ]:
# Analyze how the RL agent allocates across different ETFs
rl_weights = model_simulations['RL_GlobalEquity']['backtest_weights']
if hasattr(rl_weights, 'collect'):
    rl_weights = rl_weights.collect()

# Calculate average allocation to each ETF
print("\nAverage Allocation by RL Agent:")
for ticker in trade_tickers:
    if ticker in rl_weights.columns:
        avg_weight = rl_weights.select(ticker).mean().to_numpy()[0, 0]
        print(f"{ticker:10s}: {avg_weight*100:>6.2f}%")

# Plot allocation over time
fig, ax = plt.subplots(figsize=(14, 7))
dates = rl_weights.select('date')['date'].to_list()

# Create stacked area chart
bottom = np.zeros(len(dates))
for ticker in trade_tickers:
    if ticker in rl_weights.columns:
        weights = rl_weights.select(ticker).to_numpy().flatten()
        ax.fill_between(range(len(dates)), bottom, bottom + weights, label=ticker, alpha=0.7)
        bottom += weights

ax.set_title("RL Agent Allocation Over Time", fontsize=14, fontweight='bold')
ax.set_xlabel("Time")
ax.set_ylabel("Portfolio Weight")
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Save Trained Model

In [ ]:
# Save the trained agent for future use
torch.save({
    'policy_state_dict': trained_agent.policy.state_dict(),
    'value_state_dict': trained_agent.value.state_dict(),
    'trade_tickers': trade_tickers,
    'signal_tickers': signal_tickers,
    'momentum_columns': momentum_columns,
}, 'rl_global_equity_model.pth')

print("\nModel saved to 'rl_global_equity_model.pth'")